# Masār judge calibration on open Arabic models

Runs `pipeline/calibrate_judges.py` from [masar-platform](https://github.com/MostafaFathy1527/masar-platform)
on two open models served by llama.cpp, with the same prompt, parsing and gate as the Bedrock judges:

- **QCRI/Fanar-2-27B-Instruct** (Apache-2.0): an Arabic-English model continually pretrained from Gemma 3 27B.
  Its thinking mode is switched off, so it answers directly, as every other judge does.
- **Gemma 3 27B IT**, its base model, as a control at the same quantisation. On Bedrock, Gemma 3 27B gave all
  four lessons with English pasted into their Arabic fields the same score as the real lessons.

Both run as Q4_K_M GGUF at temperature 0.

**Before running:** in the right-hand panel, set *Accelerator* to **GPU T4 x2** and switch **Internet on**
(Kaggle asks for a verified phone number first).

**Run:** *Save Version* → **Save & Run All (Commit)**. It takes about three hours and needs no attention.

**Output:** `masar-open-judges-results.zip` in the version's *Output* tab.

Not tested here: ALLaM-7B-Instruct and Fanar-1-9B. Their 4,096-token context is shorter than the judge
prompt, which runs from about 5,500 to 8,000 tokens before the model writes anything.

In [ ]:
import datetime, glob, json, os, pathlib, shutil, subprocess, sys, time, urllib.request

REPO = 'https://github.com/MostafaFathy1527/masar-platform'
WORK = pathlib.Path('/tmp/masar')          # the repository clone
LLAMA = pathlib.Path('/tmp/llama.cpp')     # llama.cpp source and build
MODELS_DIR = pathlib.Path('/tmp/models')   # GGUF files, each deleted after its run
TEMPLATES = pathlib.Path('/tmp/templates')
OUT = pathlib.Path('/kaggle/working')      # only results are written here
CTX = 16384                                # judge prompt (up to ~8k tokens) + verdict (up to 4k)
PORT = 8080

# (served name = GGUF file name, Hugging Face repository, chat format)
MODELS = [
    ('Fanar-2-27B-Instruct.Q4_K_M.gguf', 'mradermacher/Fanar-2-27B-Instruct-GGUF', 'fanar'),
    ('gemma-3-27b-it-Q4_K_M.gguf', 'unsloth/gemma-3-27b-it-GGUF', 'gemma'),
]


def sh(cmd, cwd=None, tail=3000):
    """Run a shell command, print the end of its output, stop the notebook if it fails."""
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(((r.stdout or '') + (r.stderr or ''))[-tail:], flush=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout


started = datetime.datetime.now(datetime.timezone.utc)
print('started', started.isoformat())

In [ ]:
gpu = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
assert len(gpu.strip().splitlines()) >= 2, 'Set Accelerator to GPU T4 x2: a 16.5 GB model does not fit on one 16 GB card.'

shutil.rmtree(WORK, ignore_errors=True)
sh(f'git clone --depth 1 {REPO} {WORK}')
masar_commit = sh('git rev-parse HEAD', cwd=WORK).strip()
print('masar-platform at', masar_commit)

In [ ]:
# llama.cpp, built for the T4 (compute capability 7.5). About ten minutes.
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
server_bin = LLAMA / 'build' / 'bin' / 'llama-server'
if not server_bin.exists():
    shutil.rmtree(LLAMA, ignore_errors=True)
    sh(f'git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA}')
    # Kaggle's image has no CUDA driver library for CMake to link against (CUDA::cuda_driver
    # not found). GGML_CUDA_NO_VMM=ON drops that link; the driver is still used at run time.
    # If a libcuda is present anyway, it is passed explicitly as well.
    drivers = (sorted(glob.glob('/usr/local/cuda*/**/stubs/libcuda.so', recursive=True))
               + sorted(glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*'))
               + sorted(glob.glob('/usr/local/nvidia/lib64/libcuda.so*')))
    print('libcuda candidates:', drivers)
    driver = f' -DCUDA_cuda_driver_LIBRARY={drivers[0]}' if drivers else ''
    sh(f'cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 '
       f'-DGGML_CUDA_NO_VMM=ON -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release' + driver)
    sh(f'cmake --build {LLAMA}/build --config Release -j {os.cpu_count()} --target llama-server')
llama_commit = sh('git rev-parse HEAD', cwd=LLAMA).strip()
print('llama.cpp at', llama_commit)

In [ ]:
from huggingface_hub import hf_hub_download

try:
    import jinja2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jinja2'], check=True)
    import jinja2

# Chat formats, written out rather than left to whatever a GGUF file carries, and checked against
# the model makers' own templates before anything runs.
# Fanar-2: QCRI's template has a raw block that llama.cpp's template engine cannot parse, so the
# server gets the turn format that template renders for one user message, written plainly:
#   - no literal <bos>, because llama.cpp adds BOS itself and a doubled BOS degrades Gemma-family models;
#   - thinking off (an empty think block), so the model answers directly instead of spending the
#     4,000-token budget on a reasoning trace, as every other judge answers directly.
# Gemma 3: its standard turn format.
TEMPLATES.mkdir(parents=True, exist_ok=True)
EXPECTED = {
    'fanar': '<start_of_turn>user\nPROMPT<end_of_turn>\n<start_of_turn>model\n<think>\n\n</think>\n\n',
    'gemma': '<start_of_turn>user\nPROMPT<end_of_turn>\n<start_of_turn>model\n',
}
TEMPLATE_TEXT = {
    'fanar': ("{%- for m in messages -%}"
              "{%- if m['role'] == 'assistant' -%}{{ m['content'] + '<end_of_turn>\\n' }}"
              "{%- else -%}{{ '<start_of_turn>user\\n' + m['content'] + '<end_of_turn>\\n<start_of_turn>model\\n' }}"
              "{%- endif -%}{%- endfor -%}"
              "{{ '<think>\\n\\n</think>\\n\\n' }}"),
    'gemma': ("{%- for m in messages -%}"
              "{%- if m['role'] == 'assistant' -%}{{ '<start_of_turn>model\\n' + m['content'] + '<end_of_turn>\\n' }}"
              "{%- else -%}{{ '<start_of_turn>user\\n' + m['content'] + '<end_of_turn>\\n' }}"
              "{%- endif -%}{%- endfor -%}"
              "{%- if add_generation_prompt -%}{{ '<start_of_turn>model\\n' }}{%- endif -%}"),
}
env = jinja2.Environment()
msgs = [{'role': 'user', 'content': 'PROMPT'}]
official = pathlib.Path(hf_hub_download('QCRI/Fanar-2-27B-Instruct', 'chat_template.jinja', local_dir=str(TEMPLATES)))
qcri = env.from_string(official.read_text(encoding='utf-8')).render(
    messages=msgs, add_generation_prompt=True, tools=None, no_thinking=True)
assert qcri == '<bos>' + EXPECTED['fanar'], f'QCRI template renders {qcri!r}; review before running'
for fmt, text in TEMPLATE_TEXT.items():
    rendered = env.from_string(text).render(messages=msgs, add_generation_prompt=True)
    assert rendered == EXPECTED[fmt], f'{fmt} template renders {rendered!r}'
    (TEMPLATES / f'{fmt}.jinja').write_text(text, encoding='utf-8')
print("templates checked: the Fanar format matches QCRI's own template with thinking off")

In [ ]:
def post(path, payload, timeout=60):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}{path}', data=json.dumps(payload).encode(),
                                 headers={'Content-Type': 'application/json'}, method='POST')
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode())


def serve(gguf, fmt):
    """Start llama-server on one GGUF and check the exact prompt format before any judgement."""
    log_path = OUT / f'server-{pathlib.Path(gguf).stem}.log'
    cmd = [str(server_bin), '-m', str(gguf), '-ngl', '999', '-c', str(CTX), '-np', '1',
           '--host', '127.0.0.1', '--port', str(PORT),
           '--jinja', '--chat-template-file', str(TEMPLATES / f'{fmt}.jinja')]
    print(' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=open(log_path, 'w'), stderr=subprocess.STDOUT)
    for _ in range(180):  # loading 16 GB onto two cards takes a few minutes
        time.sleep(10)
        if proc.poll() is not None:
            raise RuntimeError('llama-server exited:\n' + log_path.read_text()[-3000:])
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=5) as resp:
                if resp.status == 200:
                    break
        except Exception:
            continue
    else:
        proc.terminate()
        raise RuntimeError('llama-server never became healthy')

    rendered = post('/apply-template', {'messages': [{'role': 'user', 'content': 'PROMPT'}]})['prompt']
    print('chat format:', repr(rendered), flush=True)
    assert rendered == EXPECTED[fmt], f'unexpected chat format for {fmt}'
    # One short request through the same path the judge uses; llama.cpp warns in its log if the
    # prompt ends up with two BOS tokens.
    reply = post('/v1/chat/completions', {'model': 'check', 'max_tokens': 8, 'temperature': 0,
                                          'messages': [{'role': 'user', 'content': 'Reply with one word: ready'}]},
                 timeout=600)
    print('test reply:', repr(reply['choices'][0]['message'].get('content')), flush=True)
    assert '2 BOS tokens' not in log_path.read_text(), 'doubled BOS: see the server log'
    return proc


def calibrate(name):
    cmd = [sys.executable, 'pipeline/calibrate_judges.py',
           '--endpoint', f'http://127.0.0.1:{PORT}/v1', '--models', name]
    print(' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    return p.wait()

In [ ]:
# Four sample lessons x (the real lesson + five broken copies) = 24 judgements per model.
MODELS_DIR.mkdir(parents=True, exist_ok=True)
status = {}
for name, repo, fmt in MODELS:
    t0 = time.time()
    gguf = hf_hub_download(repo, name, local_dir=str(MODELS_DIR))
    print(f'downloaded {name} in {time.time() - t0:.0f}s', flush=True)
    proc = serve(gguf, fmt)
    try:
        status[name] = calibrate(name)
    finally:
        proc.terminate()
        proc.wait(timeout=120)
        pathlib.Path(gguf).unlink(missing_ok=True)
    print(f'{name}: exit {status[name]} after {(time.time() - t0) / 60:.0f} min', flush=True)

In [ ]:
# Everything the run changed in the repository, at the same paths, plus how it was run.
res = OUT / 'results'
shutil.rmtree(res, ignore_errors=True)
res.mkdir()
changed = sh('git status --porcelain --untracked-files=all', cwd=WORK, tail=200)
(res / 'changed-files.txt').write_text(changed)
for line in changed.splitlines():
    rel = line[3:].strip()
    if (WORK / rel).is_file():
        (res / 'repo' / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(WORK / rel, res / 'repo' / rel)
for log in OUT.glob('server-*.log'):
    shutil.copy2(log, res / log.name)
(res / 'environment.json').write_text(json.dumps({
    'masar_commit': masar_commit, 'llama_cpp_commit': llama_commit,
    'gpu': gpu.strip().splitlines(), 'models': MODELS, 'context': CTX, 'temperature': 0,
    'exit_codes': status, 'started_utc': started.isoformat(),
    'finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}, indent=2) + '\n')
shutil.make_archive(str(OUT / 'masar-open-judges-results'), 'zip', res)
print('wrote', OUT / 'masar-open-judges-results.zip')

for report in sorted((res / 'repo').rglob('report.md')):
    print('\n' + str(report.relative_to(res / 'repo')))
    for row in report.read_text(encoding='utf-8').splitlines():
        if row.startswith('| `') and ('Fanar' in row or 'gemma-3-27b-it-Q4' in row) and ' of ' in row:
            print(row)

## After the run

Download `masar-open-judges-results.zip` from the *Output* tab. Inside, `repo/` holds every file the run
added or changed, at its path in the repository; `changed-files.txt` lists them, `environment.json` records
the commits, cards, models and settings, and `server-*.log` are llama.cpp's logs.